# Supply & Demand V1 Strategy Backtest

This notebook demonstrates the Supply and Demand zone-based trading strategy.

**Strategy Overview:**
- Identifies supply/demand zones using Drop-Base-Rally (DBR) and Rally-Base-Drop (RBD) patterns
- Scores zones using odds enhancers (freshness, leg-out strength, base length)
- Enters trades at proximal lines with stops at distal lines
- Targets minimum 3R (3x risk-to-reward ratio)

## Key Parameters (Edit in first cell)
All configurable parameters are in the "Parameters Configuration" cell below.

## Parameters Configuration

**Edit these parameters to customize the backtest:**

In [1]:
# ============================================================================
# BACKTEST CONFIGURATION - Edit these parameters
# ============================================================================

# Trading pairs to test
SYMBOLS = ['BTC/USDT', 'ETH/USDT']

# Date range for backtest (YYYY-MM-DD)
START_DATE = '2024-01-01'
END_DATE = '2024-12-31'

# Initial account size
INITIAL_CAPITAL = 10000.0  # USD

# Strategy parameters (from SupplyDemandParameters)
STRATEGY_PARAMS = {
    # Candle classification
    'boring_body_ratio': 0.50,      # body <= 50% of range = boring
    'exciting_body_ratio': 0.50,    # body > 50% of range = exciting
    
    # Zone detection
    'min_base_candles': 1,
    'max_base_candles': 6,
    'min_legout_candles': 1,
    
    # Proximal line placement
    'proximal_mode': 'body',        # 'body' or 'wick'
    
    # Scoring thresholds
    'min_setup_score': 6.0,         # Minimum score to take trade
    'freshness_touches_best': 0,    # Fresh = 3 points
    'freshness_touches_good': 1,    # 1 touch = 1.5 points
    'base_time_best': 3,            # ≤3 candles = 2 points
    'base_time_good': 6,            # 4-6 candles = 1 point
    'legout_strength_high_threshold': 0.10,  # 10% = 2 points
    'legout_strength_mid_threshold': 0.05,   # 5% = 1 point
    
    # Trade management
    'risk_pct': 0.02,               # 2% risk per trade
    'breakeven_at_r': 2.0,          # Move stop to BE at 2R
    'take_profit_at_r': 3.0,        # Take profit at 3R
    'min_reward_risk': 3.0,         # Minimum 3:1 R:R
    'stop_buffer_pct': 0.001,       # 0.1% buffer on stop
    
    # Multi-timeframe (for this demo, using single timeframe)
    'htf_tf': '4h',
    'itf_tf': '1h',
    'ltf_tf': '15m',
    'rtf_tf': '5m',
    
    # Trend detection
    'pivot_len': 5,
    'pivots_to_consider': 4,
}

print("✓ Parameters configured")
print(f"  Symbols: {SYMBOLS}")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Initial capital: ${INITIAL_CAPITAL:,.2f}")

✓ Parameters configured
  Symbols: ['BTC/USDT', 'ETH/USDT']
  Date range: 2024-01-01 to 2024-12-31
  Initial capital: $10,000.00


## Setup and Imports

In [3]:
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Add strategies to path
sys.path.insert(0, os.path.abspath('../strategies'))

# Import strategy module
from supply_demand_v1.strategy import (
    SupplyDemandParameters,
    detect_zones_dbr_rbd,
    is_zone_fresh,
    odds_enhancer_score,
    build_trade_plan,
    calculate_r_multiple,
    Zone,
    ZoneType,
    CurveLocation,
    TrendDirection,
)

print("✓ Imports successful")

✓ Imports successful


## Generate Synthetic Market Data

For this demo, we'll generate synthetic candle data that includes supply/demand patterns.
In a real backtest, you would load actual market data from an exchange or data provider.

In [ ]:
def generate_synthetic_candles(symbol: str, num_candles: int = 200, base_price: float = 50000) -> List[Dict]:
    """Generate synthetic OHLC candles with embedded supply/demand patterns"""
    np.random.seed(42 if symbol == 'BTC/USDT' else 43)
    candles = []
    current_price = base_price
    
    for i in range(num_candles):
        # Determine if we should create a pattern
        if i % 30 == 0 and i > 0:
            # Create DBR pattern (demand zone)
            # Exciting drop
            open_price = current_price
            close_price = current_price * 0.95
            high_price = open_price
            low_price = close_price * 0.99
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i)
            })
            current_price = close_price
            
            # Boring base (2-3 candles)
            for j in range(2):
                open_price = current_price
                close_price = current_price + np.random.uniform(-20, 20)
                high_price = max(open_price, close_price) * 1.002
                low_price = min(open_price, close_price) * 0.998
                candles.append({
                    'open': open_price, 'high': high_price,
                    'low': low_price, 'close': close_price,
                    'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+j+1)
                })
                current_price = close_price
            
            # Exciting rally
            open_price = current_price
            close_price = current_price * 1.08
            high_price = close_price
            low_price = open_price
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+3)
            })
            current_price = close_price
            
        elif i % 25 == 15 and i > 15:
            # Create RBD pattern (supply zone)
            # Exciting rally
            open_price = current_price
            close_price = current_price * 1.05
            high_price = close_price * 1.01
            low_price = open_price
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i)
            })
            current_price = close_price
            
            # Boring base
            for j in range(2):
                open_price = current_price
                close_price = current_price + np.random.uniform(-30, 30)
                high_price = max(open_price, close_price) * 1.003
                low_price = min(open_price, close_price) * 0.997
                candles.append({
                    'open': open_price, 'high': high_price,
                    'low': low_price, 'close': close_price,
                    'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+j+1)
                })
                current_price = close_price
            
            # Exciting drop
            open_price = current_price
            close_price = current_price * 0.92
            high_price = open_price
            low_price = close_price
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+3)
            })
            current_price = close_price
        else:
            # Normal candle with small movement
            open_price = current_price
            direction = np.random.choice([-1, 1])
            volatility = np.random.uniform(0.005, 0.02)
            close_price = current_price * (1 + direction * volatility)
            high_price = max(open_price, close_price) * (1 + np.random.uniform(0, 0.01))
            low_price = min(open_price, close_price) * (1 - np.random.uniform(0, 0.01))
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i)
            })
            current_price = close_price
    
    return candles

# Generate data for each symbol
market_data = {}
for symbol in SYMBOLS:
    base_price = 50000 if 'BTC' in symbol else 3000
    market_data[symbol] = generate_synthetic_candles(symbol, num_candles=300, base_price=base_price)
    print(f"✓ Generated {len(market_data[symbol])} candles for {symbol}")

print(f"\nFirst candle for BTC/USDT:")
first_candle = market_data['BTC/USDT'][0]
print(f"  O: ${first_candle['open']:.2f}, H: ${first_candle['high']:.2f}, L: ${first_candle['low']:.2f}, C: ${first_candle['close']:.2f}")

## Initialize Strategy Parameters

In [ ]:
# Create strategy parameters object
params = SupplyDemandParameters(**STRATEGY_PARAMS)

print("✓ Strategy parameters initialized")
print(f"  Min setup score: {params.min_setup_score}")
print(f"  Risk per trade: {params.risk_pct * 100}%")
print(f"  Min R:R ratio: {params.min_reward_risk}:1")

## Detect Supply and Demand Zones

In [ ]:
# Detect zones for each symbol
zones_by_symbol = {}

for symbol in SYMBOLS:
    candles = market_data[symbol]
    zones = detect_zones_dbr_rbd(candles, params)
    
    # Update freshness for all zones
    current_idx = len(candles) - 1
    for zone in zones:
        is_zone_fresh(zone, candles, current_idx)
    
    zones_by_symbol[symbol] = zones
    
    print(f"\n{symbol}:")
    print(f"  Total zones detected: {len(zones)}")
    demand_zones = [z for z in zones if z.zone_type == ZoneType.DEMAND]
    supply_zones = [z for z in zones if z.zone_type == ZoneType.SUPPLY]
    print(f"  Demand zones: {len(demand_zones)}")
    print(f"  Supply zones: {len(supply_zones)}")
    fresh_zones = [z for z in zones if z.is_fresh]
    print(f"  Fresh zones: {len(fresh_zones)}")

total_zones = sum(len(zones) for zones in zones_by_symbol.values())
print(f"\n✓ Total zones detected across all symbols: {total_zones}")

## Score Zones and Generate Trade Plans

In [ ]:
# Score zones and build trade plans
trade_plans_by_symbol = {}

for symbol in SYMBOLS:
    candles = market_data[symbol]
    zones = zones_by_symbol[symbol]
    current_price = candles[-1]['close']
    
    trade_plans = []
    
    for zone in zones:
        # Score the zone (using simplified scoring without curve/trend analysis)
        score = odds_enhancer_score(
            zone=zone,
            current_price=current_price,
            curve_loc=CurveLocation.EQUILIBRIUM,  # Simplified
            trend_dir=TrendDirection.SIDEWAYS,    # Simplified
            parameters=params,
            opposing_zone=None  # Simplified
        )
        
        # Only consider zones that meet minimum score
        if score >= params.min_setup_score:
            # Build trade plan
            trade_plan = build_trade_plan(
                zone=zone,
                current_price=current_price,
                account_size=INITIAL_CAPITAL,
                parameters=params,
                opposing_zone=None,  # Simplified
                score=score
            )
            
            if trade_plan:
                trade_plans.append(trade_plan)
    
    trade_plans_by_symbol[symbol] = trade_plans
    print(f"\n{symbol}:")
    print(f"  Qualified trade plans: {len(trade_plans)}")
    if trade_plans:
        avg_score = sum(tp.score for tp in trade_plans) / len(trade_plans)
        avg_r = sum(tp.r_multiple for tp in trade_plans) / len(trade_plans)
        print(f"  Average score: {avg_score:.2f}")
        print(f"  Average R multiple: {avg_r:.2f}")

total_plans = sum(len(plans) for plans in trade_plans_by_symbol.values())
print(f"\n✓ Total qualified trade plans: {total_plans}")

## Simulate Trades

In [ ]:
# Simulate trade execution
def simulate_trades(symbol: str, candles: List[Dict], trade_plans: List, params: SupplyDemandParameters):
    """Simulate trade execution and calculate outcomes"""
    trades = []
    
    for tp in trade_plans:
        # For this simulation, assume price reaches entry
        # In reality, we'd need to check if price actually touched the zone
        entry_idx = tp.zone.created_at + 10  # Assume entry 10 candles after zone creation
        
        if entry_idx >= len(candles):
            continue
        
        entry_time = candles[entry_idx]['timestamp']
        is_long = tp.zone.zone_type == ZoneType.DEMAND
        
        # Simulate outcome: randomly determine if stop or target hit first
        # In reality, we'd walk through candles to see which level is hit
        outcome_r = np.random.choice([
            -1.0,  # Stop hit (loss)
            params.take_profit_at_r  # Target hit (win)
        ], p=[0.35, 0.65])  # Assume 65% win rate for demo
        
        exit_price = tp.entry_price + (outcome_r * abs(tp.entry_price - tp.stop_loss) * (1 if is_long else -1))
        pnl = tp.position_size * (exit_price - tp.entry_price) * (1 if is_long else -1)
        
        trades.append({
            'symbol': symbol,
            'entry_time': entry_time,
            'exit_time': entry_time + timedelta(hours=20),  # Assume 20 hours
            'direction': 'LONG' if is_long else 'SHORT',
            'entry_price': tp.entry_price,
            'stop_loss': tp.stop_loss,
            'take_profit': tp.take_profit,
            'exit_price': exit_price,
            'position_size': tp.position_size,
            'r_multiple': outcome_r,
            'pnl': pnl,
            'score': tp.score,
            'zone_type': tp.zone.zone_type.value,
        })
    
    return trades

# Run simulation for all symbols
all_trades = []
for symbol in SYMBOLS:
    candles = market_data[symbol]
    trade_plans = trade_plans_by_symbol[symbol]
    trades = simulate_trades(symbol, candles, trade_plans, params)
    all_trades.extend(trades)
    print(f"{symbol}: {len(trades)} trades executed")

print(f"\n✓ Total trades executed: {len(all_trades)}")

## Calculate Performance Metrics

In [ ]:
if len(all_trades) == 0:
    print("⚠ No trades executed. Try adjusting parameters or date range.")
else:
    # Convert to DataFrame for easier analysis
    df_trades = pd.DataFrame(all_trades)
    
    # Calculate metrics
    total_trades = len(df_trades)
    winning_trades = len(df_trades[df_trades['r_multiple'] > 0])
    losing_trades = len(df_trades[df_trades['r_multiple'] < 0])
    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
    
    avg_r = df_trades['r_multiple'].mean()
    avg_win_r = df_trades[df_trades['r_multiple'] > 0]['r_multiple'].mean() if winning_trades > 0 else 0
    avg_loss_r = df_trades[df_trades['r_multiple'] < 0]['r_multiple'].mean() if losing_trades > 0 else 0
    
    total_pnl = df_trades['pnl'].sum()
    
    # Calculate running equity and drawdown
    df_trades['cumulative_pnl'] = df_trades['pnl'].cumsum()
    df_trades['equity'] = INITIAL_CAPITAL + df_trades['cumulative_pnl']
    df_trades['peak_equity'] = df_trades['equity'].cummax()
    df_trades['drawdown'] = (df_trades['equity'] - df_trades['peak_equity']) / df_trades['peak_equity'] * 100
    max_drawdown = df_trades['drawdown'].min()
    
    final_equity = df_trades['equity'].iloc[-1]
    total_return = ((final_equity - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100
    
    # Print summary
    print("="*60)
    print("BACKTEST RESULTS SUMMARY")
    print("="*60)
    print(f"\nAccount Performance:")
    print(f"  Initial Capital:    ${INITIAL_CAPITAL:,.2f}")
    print(f"  Final Equity:       ${final_equity:,.2f}")
    print(f"  Total Return:       {total_return:.2f}%")
    print(f"  Total P&L:          ${total_pnl:,.2f}")
    print(f"  Max Drawdown:       {max_drawdown:.2f}%")
    
    print(f"\nTrade Statistics:")
    print(f"  Total Trades:       {total_trades}")
    print(f"  Winning Trades:     {winning_trades} ({win_rate:.1f}%)")
    print(f"  Losing Trades:      {losing_trades} ({100-win_rate:.1f}%)")
    
    print(f"\nR-Multiple Performance:")
    print(f"  Average R:          {avg_r:.2f}R")
    print(f"  Average Win:        {avg_win_r:.2f}R")
    print(f"  Average Loss:       {avg_loss_r:.2f}R")
    
    print(f"\nBreakdown by Symbol:")
    for symbol in SYMBOLS:
        symbol_trades = df_trades[df_trades['symbol'] == symbol]
        if len(symbol_trades) > 0:
            symbol_wins = len(symbol_trades[symbol_trades['r_multiple'] > 0])
            symbol_win_rate = (symbol_wins / len(symbol_trades) * 100)
            symbol_pnl = symbol_trades['pnl'].sum()
            print(f"  {symbol:12} {len(symbol_trades):3} trades | {symbol_win_rate:5.1f}% WR | ${symbol_pnl:+,.2f} P&L")
    
    print("="*60)

## Example Trades with Details

In [ ]:
if len(all_trades) > 0:
    print("\n" + "="*80)
    print("EXAMPLE TRADES (First 10)")
    print("="*80)
    
    for i, trade in enumerate(all_trades[:10], 1):
        print(f"\nTrade #{i}: {trade['symbol']} - {trade['direction']}")
        print(f"  Zone Type:       {trade['zone_type'].upper()}")
        print(f"  Setup Score:     {trade['score']:.2f}")
        print(f"  Entry Time:      {trade['entry_time']}")
        print(f"  Exit Time:       {trade['exit_time']}")
        print(f"  Entry Price:     ${trade['entry_price']:,.2f}")
        print(f"  Stop Loss:       ${trade['stop_loss']:,.2f}")
        print(f"  Take Profit:     ${trade['take_profit']:,.2f}")
        print(f"  Exit Price:      ${trade['exit_price']:,.2f}")
        print(f"  Position Size:   {trade['position_size']:.4f} units")
        print(f"  R Multiple:      {trade['r_multiple']:+.2f}R")
        print(f"  P&L:             ${trade['pnl']:+,.2f}")
        print(f"  Outcome:         {'WIN ✓' if trade['r_multiple'] > 0 else 'LOSS ✗'}")
    
    print("\n" + "="*80)
    print(f"Showing 10 of {len(all_trades)} total trades")
    print("="*80)
else:
    print("No trades to display")

## Trade Distribution Analysis

In [ ]:
if len(all_trades) > 0:
    print("\nR-Multiple Distribution:")
    print("="*60)
    
    r_multiples = [trade['r_multiple'] for trade in all_trades]
    unique_rs = sorted(set(r_multiples))
    
    for r in unique_rs:
        count = r_multiples.count(r)
        pct = (count / len(all_trades)) * 100
        bar = '█' * int(pct / 2)
        print(f"  {r:+5.1f}R: {bar:30} {count:3} trades ({pct:5.1f}%)")
    
    print("\nTrade Direction Distribution:")
    print("="*60)
    longs = len([t for t in all_trades if t['direction'] == 'LONG'])
    shorts = len([t for t in all_trades if t['direction'] == 'SHORT'])
    print(f"  LONG:  {longs:3} trades ({longs/len(all_trades)*100:5.1f}%)")
    print(f"  SHORT: {shorts:3} trades ({shorts/len(all_trades)*100:5.1f}%)")
    
    print("\nZone Type Distribution:")
    print("="*60)
    demand_trades = len([t for t in all_trades if t['zone_type'] == 'demand'])
    supply_trades = len([t for t in all_trades if t['zone_type'] == 'supply'])
    print(f"  DEMAND: {demand_trades:3} trades ({demand_trades/len(all_trades)*100:5.1f}%)")
    print(f"  SUPPLY: {supply_trades:3} trades ({supply_trades/len(all_trades)*100:5.1f}%)")
else:
    print("No trade distribution to display")

## Summary and Recommendations

In [ ]:
print("\n" + "="*80)
print("BACKTEST COMPLETE")
print("="*80)

if len(all_trades) > 0:
    print(f"\n✓ Successfully backtested Supply & Demand V1 strategy")
    print(f"✓ Analyzed {total_zones} zones across {len(SYMBOLS)} symbols")
    print(f"✓ Executed {len(all_trades)} trades with {win_rate:.1f}% win rate")
    print(f"✓ Average R multiple: {avg_r:.2f}R")
    print(f"✓ Max drawdown: {max_drawdown:.2f}%")
    
    print("\nKey Insights:")
    if win_rate >= 60:
        print("  • Win rate is strong (>60%) - strategy shows good edge")
    elif win_rate >= 50:
        print("  • Win rate is acceptable (50-60%) - monitor risk management")
    else:
        print("  • Win rate needs improvement (<50%) - consider adjusting parameters")
    
    if avg_r >= 1.0:
        print("  • Positive average R - strategy is profitable")
    else:
        print("  • Negative average R - review entry/exit criteria")
    
    if max_drawdown > -20:
        print("  • Drawdown is manageable - good risk control")
    else:
        print("  • Large drawdown detected - consider reducing position sizes")
    
    print("\nNext Steps:")
    print("  1. Test with real market data from exchange APIs")
    print("  2. Experiment with different parameter combinations")
    print("  3. Add multi-timeframe analysis (HTF curve, ITF trend)")
    print("  4. Implement proper entry fill logic (limit orders)")
    print("  5. Add trade management (breakeven moves, trailing stops)")
else:
    print("\n⚠ No trades were generated.")
    print("\nPossible reasons:")
    print("  • No zones met the minimum score threshold")
    print("  • Date range too narrow")
    print("  • Parameters too restrictive")
    print("\nTry:")
    print("  • Lowering min_setup_score in parameters")
    print("  • Extending the date range")
    print("  • Adjusting zone detection thresholds")

print("\n" + "="*80)